# KernelPack RAG — Retrieval Experiments (v1→v3)

## TL;DR
- Baseline hybrid retrieval (UniXcoder + BM25 + RRF) achieves 20% recall@3
  on 10 solvers Q&A pairs.
- Wide retrieval diagnosis (top-50) reveals the real failure mode is ranking,
  not vocabulary gap — 9/10 targets land in the top-10 neighborhood after
  adding LLM summaries, but fall outside top-3.
- LLM summaries slightly bump recall to 30%. A cross-encoder reranker is the
  highest-leverage next step.

## What this notebook does
1. Builds a baseline hybrid RAG pipeline on kernelpack-python (v1)
2. Evaluates retrieval quality against 10 solvers Q&A pairs
3. Diagnoses failure modes per-miss using wide retrieval (top-50) —
   classifying each as not-indexed, vocabulary gap, one-hop miss, or
   ranking problem
4. Tests two improvements (v2, v3), measures impact, and re-diagnoses
   after each change

## Results at a glance
| Version | Change | recall@3 |
|---------|--------|----------|
| v1 | Baseline — raw source, MIN_LINES=5 | 2/10 (20%) |
| v2 | MIN_LINES=1 | 1/10 (10%) — worse |
| v3 | LLM summaries prepended to solver chunks | 3/10 (30%) |

Note: v3 is scoped to solvers module only. Full-index comparison is a next step.

## Key findings
- Vocabulary gap is real but not the primary bottleneck. After summaries,
  9/10 targets appear in the top-10 retrieved neighborhood.
- The bottleneck is ranking — correct chunks exist in the neighborhood
  but fall outside top-3. A cross-encoder reranker applied post-retrieval
  would plausibly recover 7-8/10.
- MIN_LINES filtering drops genuine targets (set_nonlinear_tolerance is
  2 lines) but lowering it adds noise that degrades the whole embedding
  space. Line count is the wrong filter — semantic content is the right one.
- One-hop retrieval is real and measurable. PUDiffusionSolver.init was
  not indexed but referenced in retrieved neighbors — cross-reference
  metadata would recover this class of miss.

## What's not in this notebook
- Summaries for non-solvers modules (geometry, nodes, domain, poly, rbffd)
- Fair v3 comparison against full 273-chunk index
- Cross-encoder reranker (identified as highest-leverage next step)

## Notebook Background

### Eval set
- **Source:** `qa_pairs/solvers_qa.json` — 10 Q&A pairs covering kernelpack.solvers
- **Generated by:** Codex, prompted to ingest the full repo and produce pairs
  that sound like questions from a scientist who knows what they want numerically
  but doesn't know KernelPack's API yet
- **Tiers:** api (5), workflow (2), conceptual (3)
- **Validation status:** Codex-generated, requires more intensive review. Ground truth
  is unconfirmed — treat recall numbers as directional only.

Carried over from naive-rag.ipynb (section 4):
- tree-sitter AST chunking at function boundaries
- UniXcoder dense embeddings via ChromaDB
- BM25 sparse leg (whitespace-tokenized)
- RRF fusion (k=60), top 3 returned

### How experiments are structured
- Each version isolates one change, runs the same 10-query eval, and
  compares recall@3 against v1 before combining changes.
- Failure modes are diagnosed per-miss before and after each change
  using wide retrieval (top-50) — no blind knob-turning.

## v1 — Baseline Pipeline

### Chunking
- Tool: tree-sitter (Python grammar)
- Unit: function_definition nodes only — class and module docstrings excluded
- Filter: MIN_LINES=5, inclusive (functions with fewer than 5 lines dropped)
- Off-by-one corrected from naive-rag.ipynb — threshold is now truly inclusive
- Total chunks indexed: 273

### Index
- Store: ChromaDB (ephemeral)
- Collection: kp-hybrid-v1

### Embedding (dense leg)
- Model: UniXcoder (microsoft/unixcoder-base)
- Input: raw function source text
- Max sequence length: 512 tokens — long functions truncated silently
- Query encoding: same model applied to raw query string at retrieval time

### Retrieval
- Dense leg: UniXcoder query → ChromaDB, top 10 by L2 distance
- Sparse leg: BM25 (whitespace-tokenized), top 10 by BM25 score
- Fusion: RRF (k=60) merges both ranked lists
- Output: top 3 chunks by RRF score

### Known gaps in this design
- MIN_LINES=5 drops short but real targets — set_nonlinear_tolerance
  and set_linear_tolerance are 2-line functions, never indexed
- Chunker captures function_definition nodes only — class and module
  docstrings excluded
- Raw source only — no plain-English descriptions (vocabulary gap
  between NL queries and code identifiers)
- Whitespace BM25 tokenization misses camelCase and snake_case splits

In [1]:
%pip install -q \
    sentence-transformers==5.5.0 \
    tree-sitter==0.25.2 \
    tree-sitter-python==0.25.0 \
    chromadb==1.5.9 \
    rank-bm25==0.2.2


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from pathlib import Path

import chromadb
import tree_sitter_python as tspython
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tree_sitter import Language, Parser

# ── paths ──────────────────────────────────────────────────────────────────────
# Assumes kernelpack-python is cloned in the same parent directory as this repo.
# If not, update this path to point at your local kernelpack-python/src/kernelpack.
REPO_PATH = Path("../../kernelpack-python/src/kernelpack")
if not REPO_PATH.exists():
    raise FileNotFoundError(
        f"kernelpack-python not found at {REPO_PATH.resolve()}. "
        "Clone it as a sibling directory: git clone https://github.com/ShankarLab/kernelpack-python"
    )

# ── retrieval config ───────────────────────────────────────────────────────────
TOP_K     = 3   # results returned per query

In [3]:
# Load all .py files from the repo
def load_docs(repo_path: Path) -> list[dict]:
    docs = []
    for path in sorted(repo_path.rglob("*.py")): # will include __init__.py files
        docs.append({"path": str(path), "text": path.read_text(encoding="utf-8")})
    print(f"Loaded {len(docs)} files from {repo_path}")
    return docs

docs = load_docs(REPO_PATH)

Loaded 27 files from ../../kernelpack-python/src/kernelpack


In [4]:
# CHUNKING
MIN_LINES = 5   # functions with fewer than 5 lines dropped (inclusive)

# tree-sitter parser setup
PY_LANGUAGE = Language(tspython.language())
parser_ts   = Parser(PY_LANGUAGE)

def get_class_name(node):
    parent = node.parent
    if parent and parent.type == "decorated_definition":
        parent = parent.parent
    if (parent and parent.type == "block"
            and parent.parent and parent.parent.type == "class_definition"):
        return parent.parent.child_by_field_name("name").text.decode("utf-8")
    return None

# Updated extract_chunks: same tree-sitter walk, now drops functions < MIN_LINES
def extract_chunks(source: str, tree, path: str, min_lines: int = MIN_LINES):
    """Walk the AST; return (kept, dropped) split by min_lines threshold."""
    kept, dropped = [], []
    def walk(node):
        if node.type == "function_definition":
            start = node.start_point[0]
            end   = node.end_point[0]
            entry = {
                "path":          path,
                "function_name": node.child_by_field_name("name").text.decode("utf-8"),
                "class_name":    get_class_name(node),
                "text":          source[node.start_byte:node.end_byte],
                "start_line":    start,
                "end_line":      end,
            }
            (kept if (end - start + 1) >= min_lines else dropped).append(entry)
            return # don't recurse into functions
        for child in node.children:
            walk(child)
    walk(tree.root_node)
    return kept, dropped

all_chunks, all_dropped = [], []
for doc in docs:
    tree = parser_ts.parse(doc["text"].encode("utf-8"))
    kept, dropped = extract_chunks(doc["text"], tree, doc["path"])
    all_chunks.extend(kept)
    all_dropped.extend(dropped)

print(f"Chunks kept   : {len(all_chunks)}")
print(f"Chunks dropped: {len(all_dropped)}  (functions < {MIN_LINES} lines)")

Chunks kept   : 273
Chunks dropped: 245  (functions < 5 lines)


In [5]:
# INDEXING

# Build ChromaDB collection with filtered chunks + UniXcoder embeddings
documents, metadatas, ids = [], [], []
seen = set()
for c in all_chunks:
    fp  = str(Path(c["path"]).relative_to(REPO_PATH))
    cid = f"{fp}:{c['start_line']}-{c['end_line']}"
    if cid in seen:
        continue
    seen.add(cid)
    documents.append(c["text"])
    metadatas.append({
        "file_path":     fp,
        "class_name":    c["class_name"] or "",
        "function_name": c["function_name"],
        "start_line":    c["start_line"],
        "end_line":      c["end_line"],
    })
    ids.append(cid)

# EMBEDDING 
model = SentenceTransformer("microsoft/unixcoder-base")
model.max_seq_length = 512
ef = SentenceTransformerEmbeddingFunction(model_name="microsoft/unixcoder-base")
ef._model = model

client = chromadb.EphemeralClient()
col = client.get_or_create_collection("kp-hybrid-v1", embedding_function=ef)
col.add(documents=documents, metadatas=metadatas, ids=ids)
print(f"Collection kp-hybrid-v1: {col.count()} chunks")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Collection kp-hybrid-v1: 273 chunks


In [6]:
# BM25 index over the same filtered document set (whitespace tokenization)
tokenized_corpus = [doc.split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index built.")

BM25 index built.


In [7]:
def retrieve(query: str, k: int = TOP_K, silent: bool = False):

    # dense leg - embed with unixcoder, return top 10 by L2 distance
    dense_res = col.query(query_texts=[query], n_results=10)
    dense_ids = dense_res["ids"][0]
    dense_l2  = {cid: d for cid, d in zip(dense_res["ids"][0], dense_res["distances"][0])}

    # BM25 leg - scored by keyword overlap, return top 10 by BM25 score
    bm25_scores = bm25.get_scores(query.split())
    top_bm25    = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:10]
    bm25_ids    = [ids[i] for i in top_bm25]

    # RRF fusion — combine dense and BM25 ranked lists into one score per chunk
    # formula: score += 1 / (rank + 60) for each list the chunk appears in
    # k=60 = smoothing constant  (dampens the advantage of being ranked 1st)
    # a chunk appearing in both lists gets contributions from both
    rrf: dict[str, float] = {}
    for rank, cid in enumerate(dense_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)
    for rank, cid in enumerate(bm25_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)

    # sort chunks by RRF score (descending), keep top-k 
    top_ids = sorted(rrf, key=rrf.__getitem__, reverse=True)[:k]

    # fetch full text and metadata by ID from ChromaDB collection
    fetched  = col.get(ids=top_ids, include=["documents", "metadatas"])

    # ChromaDB ( col.get() ) returns results in the order of the input IDs -> need to reorder them by RRF score
    # build lookup from ID -> position in fetched results
    # reorder fetched results to match RRF ranking 
    id_to_idx = {cid: i for i, cid in enumerate(fetched["ids"])}
    ordered_ids   = [cid for cid in top_ids if cid in id_to_idx]
    ordered_docs  = [fetched["documents"][id_to_idx[cid]] for cid in ordered_ids]
    ordered_metas = [fetched["metadatas"][id_to_idx[cid]] for cid in ordered_ids]

    if not silent:
        print(f"\nQUERY: \"{query}\"")
        print("=" * 60)
        for i, (cid, doc, meta) in enumerate(
            zip(ordered_ids, ordered_docs, ordered_metas), 1
        ):
            cls    = f"{meta['class_name']}." if meta["class_name"] else ""
            l2_str = f"{dense_l2[cid]:.4f}" if cid in dense_l2 else "n/a (BM25-only)"
            print(f"\n  Rank {i} | RRF {rrf[cid]:.4f} | L2 {l2_str}")
            print(f"  {meta['file_path']} — {cls}{meta['function_name']} (lines {meta['start_line']}–{meta['end_line']})")
            print("  " + "-" * 56)
            print("  " + doc[:300].replace("\n", "\n  "))

    return ordered_metas

## v1 — Baseline Eval
Running recall@3 against solvers_qa.json (10 pairs) and benchmark_qa.json (3 pairs). These results are the baseline every subsequent version is compared against.

In [8]:
import json

with open("../qa_pairs/solvers_qa.json") as f:
    qa_pairs_solvers = json.load(f)

def eval_query(qa):
    results = retrieve(qa["query"], silent=True)
    target_file = qa["source_file"].split("/")[-1]
    target_sym  = qa["source_symbol"].split(".")[-1]

    # hit = target file and function name BOTH appear in top-k results (exact match)
    hit = any(
        r["file_path"].split("/")[-1] == target_file and
        r["function_name"] == target_sym
        for r in results
    )
    
    print(f"\n{'='*70}")
    print(f"QUERY : {qa['query']}")
    print(f"TIER  : {qa['tier']}")
    print(f"TARGET: {qa['source_symbol']} ({qa['source_file']})")
    print(f"ANSWER: {qa['expected_answer']}")
    print(f"{'─'*70}")
    for i, r in enumerate(results, 1):
        print(f"  Rank {i}: {r['file_path']} — {r['function_name']}")
    print(f"{'─'*70}")
    print(f"VERDICT: {'✓ HIT' if hit else '✗ MISS'}")

for qa in qa_pairs_solvers:
    eval_query(qa)


QUERY : If my Poisson problem has only Neumann boundary data, how is the constant nullspace removed?
TIER  : conceptual
TARGET: PoissonSolver.solve (src/kernelpack/solvers/poisson.py)
ANSWER: It treats the problem as pure Neumann when max(abs(dir_coeff)) <= 1e-13. The sparse system is augmented with a column and row of ones, last_solve_used_nullspace_ is set to True, full_state is sol[:-1], and lagrange_multiplier is sol[-1].
──────────────────────────────────────────────────────────────────────
  Rank 1: solvers/_common.py — build_system_matrix
  Rank 2: solvers/nonlinear_variable_poisson.py — _build_initial_unknown
  Rank 3: solvers/_common.py — build_system_rhs
──────────────────────────────────────────────────────────────────────
VERDICT: ✗ MISS

QUERY : How is the variable-coefficient operator -div(a grad u) represented in the discretization?
TIER  : conceptual
TARGET: VariablePoissonSolver.solve (src/kernelpack/solvers/variable_poisson.py)
ANSWER: It discretizes -div(a grad u). 

In [9]:
# recall@3 by tier — counts hits per query type against v1 index
hits_v1 = {"api": 0, "workflow": 0, "conceptual": 0}
totals_v1 = {"api": 0, "workflow": 0, "conceptual": 0}

for qa in qa_pairs_solvers:
    results = retrieve(qa["query"], silent=True)
    target_file = qa["source_file"].split("/")[-1]
    target_sym  = qa["source_symbol"].split(".")[-1]
    hit = any(
        r["file_path"].split("/")[-1] == target_file and
        r["function_name"] == target_sym
        for r in results
    )
    totals_v1[qa["tier"]] += 1
    if hit:
        hits_v1[qa["tier"]] += 1

print("\n── Recall@3 by tier ──────────────────")
for tier in ["api", "workflow", "conceptual"]:
    h, t = hits_v1[tier], totals_v1[tier]
    print(f"  {tier:<12} {h}/{t}  ({round(100 * h / t) if t else 0}%)")
total_h = sum(hits_v1.values())
total_t = sum(totals_v1.values())
print(f"  {'overall':<12} {total_h}/{total_t}  ({round(100 * total_h / total_t) if total_t else 0}%)")


── Recall@3 by tier ──────────────────
  api          1/5  (20%)
  workflow     1/2  (50%)
  conceptual   0/3  (0%)
  overall      2/10  (20%)


In [10]:
with open("../qa_pairs/benchmark_qa.json") as f:
    qa_pairs_benchmark = json.load(f)

for qa in qa_pairs_benchmark:
    eval_query(qa)

hits_bench_v1 = {"api": 0, "workflow": 0, "conceptual": 0}
totals_bench_v1 = {"api": 0, "workflow": 0, "conceptual": 0}

for qa in qa_pairs_benchmark:
    results = retrieve(qa["query"], silent=True)
    target_file = qa["source_file"].split("/")[-1]
    target_sym  = qa["source_symbol"].split(".")[-1]
    hit = any(
        r["file_path"].split("/")[-1] == target_file and
        r["function_name"] == target_sym
        for r in results
    )
    totals_bench_v1[qa["tier"]] += 1
    if hit:
        hits_bench_v1[qa["tier"]] += 1

print("\n── Recall@3 by tier ──────────────────")
for tier in ["api", "workflow", "conceptual"]:
    h, t = hits_bench_v1[tier], totals_bench_v1[tier]
    print(f"  {tier:<12} {h}/{t}  ({round(100 * h / t) if t else 0}%)")
total_h = sum(hits_bench_v1.values())
total_t = sum(totals_bench_v1.values())
print(f"  {'overall':<12} {total_h}/{total_t}  ({round(100 * total_h / total_t) if total_t else 0}%)")


QUERY : how to create an RBF-FD differentiation matrix?
TIER  : workflow
TARGET: FDDiffOp.assemble_op (src/kernelpack/rbffd/core.py)
ANSWER: Use the RBF-FD assembly classes in kernelpack.rbffd. First create stencil parameters with StencilProperties.from_accuracy(...), for example StencilProperties.from_accuracy(operator="lap", convergence_order=4, dimension=domain.get_dim(), approximation="rbf", tree_mode="all", point_set="interior_boundary"). Create operator options with OpProperties(...), then instantiate FDDiffOp(lambda: RBFStencil()) and call assemble_op(domain, op_name, st_props, op_props). For a Laplacian, use op_name="lap"; for a first derivative/gradient, use op_name="grad" and set OpProperties(selectdim=d) for the coordinate direction. Retrieve the sparse differentiation matrix with get_op(), which returns a scipy.sparse.csr_matrix. FDODiffOp has the same assemble_op API for overlapped assembly.
──────────────────────────────────────────────────────────────────────
  Rank 1: 

In [11]:
# see what got dropped during chunking
results = col.get(include=["metadatas"])
all_functions = [m["function_name"] for m in results["metadatas"]]

targets = [
    "set_nonlinear_tolerance",
    "set_periodic_boundary", 
    "set_linear_tolerance",
    "bdf1_step",
]

for target in targets:
    print(target, "→", "IN INDEX" if target in all_functions else "MISSING")

set_nonlinear_tolerance → MISSING
set_periodic_boundary → IN INDEX
set_linear_tolerance → MISSING
bdf1_step → IN INDEX


In [12]:
# ============ DIAGNOSTIC CELL =============

# for each query, check:
# 1. where does the target chunk rank in dense-only retrieval (top-50)?
# 2. is the target referenced anywhere in the top-10 chunks?
# Separates "found the right neighborhood" from "found the exact target"
for qa in qa_pairs_solvers:
    target_sym = qa["source_symbol"].split(".")[-1]
    target_file = qa["source_file"].split("/")[-1]
    
    # dense-only retrieval (ignore BM25 and RRF to isolate embedding quality)
    results_wide = col.query(query_texts=[qa["query"]], n_results=50) 
    all_metas = results_wide["metadatas"][0]
    
    # where does target chunk first appear in ranked results (if at all)?
    target_rank = next(
        (i+1 for i, m in enumerate(all_metas)
         if m["file_path"].split("/")[-1] == target_file
         and m["function_name"] == target_sym),
        None            # None if not in index at all 
    )
    
    # one-hop signal: is target symbol mentioned in any of the top-10 retrieved chunks?
    # True -> neighboring chunk calls/references the target (one-hop miss)
    # False -> target is not found in retrieved neighborhood (true miss)
    top10_docs = results_wide["documents"][0][:10]              
    referenced = any(target_sym in doc for doc in top10_docs)
    
    print(f"{qa['source_symbol']}")
    print(f"  target rank : {target_rank or 'not found'}")
    print(f"  referenced in top-10 : {referenced}")

PoissonSolver.solve
  target rank : 15
  referenced in top-10 : True
VariablePoissonSolver.solve
  target rank : 22
  referenced in top-10 : False
NonlinearVariablePoissonSolver.set_nonlinear_tolerance
  target rank : not found
  referenced in top-10 : False
DiffusionSolver.bdf3_step
  target rank : 1
  referenced in top-10 : True
PUDiffusionSolver.init
  target rank : not found
  referenced in top-10 : True
MultiSpeciesDiffusionSolver.bdf1_step
  target rank : 4
  referenced in top-10 : True
HeterogeneousMultiSpeciesDiffusionSolver.bdf1_step
  target rank : not found
  referenced in top-10 : False
PUSLAdvectionSolver.set_periodic_boundary
  target rank : 6
  referenced in top-10 : True
PUSLFDAdvectionDiffusionSolver.bdf1_step
  target rank : 1
  referenced in top-10 : True
PUSLPUAdvectionDiffusionSolver.init
  target rank : 27
  referenced in top-10 : True


## v1 — Failure Mode Analysis

Ran wide retrieval (top-50) on all 10 eval pairs to classify each miss.

| Target | Rank@50 | Referenced in top-10 | Category |
|--------|---------|----------------------|----------|
| PoissonSolver.solve | 15 | ✓ | **Reranker** problem |
| VariablePoissonSolver.solve | 22 | ✗ | Vocabulary gap |
| NonlinearVariablePoissonSolver.set_nonlinear_tolerance | not found | ✗ | Not indexed (2-line function) |
| DiffusionSolver.bdf3_step | 1 | ✓ | Hit  |
| PUDiffusionSolver.init | not found | ✓ | One-hop miss |
| MultiSpeciesDiffusionSolver.bdf1_step | 4 | ✓ | **Reranker** problem |
| HeterogeneousMultiSpeciesDiffusionSolver.bdf1_step | not found | ✗ | Vocabulary gap + not indexed |
| PUSLAdvectionSolver.set_periodic_boundary | 6 | ✓ | **Reranker** problem |
| PUSLFDAdvectionDiffusionSolver.bdf1_step | 1 | ✓ | Hit  |
| PUSLPUAdvectionDiffusionSolver.init | 26 | ✓ | **Reranker** problem |

### Key finding
- 4 misses are reranker problems 
- the target chunk exists and is referenced in the top-10 neighborhood, but falls outside top-3. 
- A cross-encoder reranker re-scores candidates by reading query and chunk together, 
not just comparing embeddings. 

### Implication for v3
LLM summaries address vocabulary gap (VariablePoissonSolver.solve). 
They cannot fix reranker problems or not-indexed chunks. Expected v3 
improvement is narrow — 1-2 additional hits at most.

## Tuning Plan

### Confirmed failure modes

1. **Not indexed** — `set_nonlinear_tolerance`, `set_linear_tolerance` are 2-line 
   functions, dropped by MIN_LINES filter. Summaries cannot help chunks that don't exist.
   - `PUDiffusionSolver.init` also not found — needs investigation (not a short function)

2. **Reranker problem** — target chunk exists and is referenced in retrieved neighborhood 
   but falls outside top-3. Affects 4 misses:
   - PoissonSolver.solve (rank 15)
   - MultiSpeciesDiffusionSolver.bdf1_step (rank 4)
   - PUSLAdvectionSolver.set_periodic_boundary (rank 6)
   - PUSLPUAdvectionDiffusionSolver.init (rank 26)

3. **Vocabulary gap** — indexed but retrieval finds wrong neighborhood entirely:
   - VariablePoissonSolver.solve (rank 22, not referenced in top-10)
   - HeterogeneousMultiSpeciesDiffusionSolver.bdf1_step (not found, not referenced)

### Knobs to tune (in order)

| Version | Change | Targets |
|---------|--------|---------|
| v2 | MIN_LINES=1 | not-indexed misses |
| v3 | LLM summaries | vocabulary gap misses |
| v4 | cross-encoder reranker | reranker-problem misses — highest leverage |

### Metric
recall@3 on the 10-item eval set (solvers_qa.json)

## v2 — MIN_LINES=1
**Change:** Drop minimum line filter from 5 to 1  
**Hypothesis:** Recovers set_nonlinear_tolerance and set_linear_tolerance

In [13]:
all_chunks_v2, all_dropped_v2 = [], []
for doc in docs:
    tree = parser_ts.parse(doc["text"].encode("utf-8"))
    kept, dropped = extract_chunks(doc["text"], tree, doc["path"], min_lines=1)
    all_chunks_v2.extend(kept)
    all_dropped_v2.extend(dropped)

print(f"v1 chunks   : {len(all_chunks)}")
print(f"v2 chunks   : {len(all_chunks_v2)}")

# re-index into new collection
col_v2 = client.get_or_create_collection("kp-hybrid-v2", embedding_function=ef)
col_v2.add(
    ids=[f"{c['path']}::{c['function_name']}::{c['start_line']}" for c in all_chunks_v2],
    documents=[c["text"] for c in all_chunks_v2],
    metadatas=[{
        "file_path":     str(Path(c["path"]).relative_to(REPO_PATH)),
        "function_name": c["function_name"],
        "class_name":    c["class_name"] or "",
        "start_line":    c["start_line"],
        "end_line":      c["end_line"],
    } for c in all_chunks_v2]
)

# verify targets now exist
all_fns_v2 = [m["function_name"] for m in col_v2.get(include=["metadatas"])["metadatas"]]
for target in ["set_nonlinear_tolerance", "set_linear_tolerance"]:
    print(target, "→", "IN INDEX" if target in all_fns_v2 else "STILL MISSING")

v1 chunks   : 273
v2 chunks   : 518
set_nonlinear_tolerance → IN INDEX
set_linear_tolerance → IN INDEX


In [14]:
# rebuild BM25 on v2 chunks
bm25_v2 = BM25Okapi([c["text"].split() for c in all_chunks_v2])
ids_v2 = [f"{c['path']}::{c['function_name']}::{c['start_line']}" for c in all_chunks_v2]

# swap col and bm25 to v2, re-run eval
col_orig, bm25_orig, ids_orig = col, bm25, ids
col, bm25, ids = col_v2, bm25_v2, ids_v2

hits_v2 = {"api": 0, "workflow": 0, "conceptual": 0}
totals_v2 = {"api": 0, "workflow": 0, "conceptual": 0}
with open("../qa_pairs/solvers_qa.json") as f:
    qa_pairs = json.load(f)
for qa in qa_pairs:
    results = retrieve(qa["query"], silent=True)
    target_file = qa["source_file"].split("/")[-1]
    target_sym  = qa["source_symbol"].split(".")[-1]
    hit = any(target_file in r["file_path"] and target_sym in r["function_name"]
              for r in results)
    totals_v2[qa["tier"]] += 1
    if hit:
        hits_v2[qa["tier"]] += 1

# restore v1
col, bm25, ids = col_orig, bm25_orig, ids_orig

print("── v1 vs v2 recall@3 ─────────────────")
print(f"  v1 overall: {sum(hits_v1.values())}/10 ({round(100 * sum(hits_v1.values()) / 10)}%)")
print(f"  v2 overall: {sum(hits_v2.values())}/10 ({sum(hits_v2.values())*10}%)")

── v1 vs v2 recall@3 ─────────────────
  v1 overall: 2/10 (20%)
  v2 overall: 1/10 (10%)


In [15]:
# which miss in v1 -> a hit in v2, and which hit -> a miss?
# retrieval using RRF (dense and BM25)
for qa in qa_pairs_solvers:
    target_file = qa["source_file"].split("/")[-1]
    target_sym  = qa["source_symbol"].split(".")[-1]
    
    # v1 result
    col, bm25, ids = col_orig, bm25_orig, ids_orig
    r1 = retrieve(qa["query"], silent=True)
    hit_v1 = any(
        r["file_path"].split("/")[-1] == target_file and
        r["function_name"] == target_sym
        for r in r1
    )
    
    # v2 result 
    col, bm25, ids = col_v2, bm25_v2, ids_v2
    r2 = retrieve(qa["query"], silent=True)
    hit_v2 = any(
        r["file_path"].split("/")[-1] == target_file and
        r["function_name"] == target_sym
        for r in r2
    )
        
    col, bm25, ids = col_orig, bm25_orig, ids_orig
    
    if hit_v1 != hit_v2:
        status = "✓ -> ✗ REGRESSED" if hit_v1 else "✗ -> ✓ RECOVERED"
        print(f"{status}: {qa['source_symbol']}")

✓ -> ✗ REGRESSED: DiffusionSolver.bdf3_step


In [16]:
col_orig, bm25_orig, ids_orig = col, bm25, ids
col, bm25, ids = col_v2, bm25_v2, ids_v2

for qa in qa_pairs_solvers:
    target_sym  = qa["source_symbol"].split(".")[-1]
    target_file = qa["source_file"].split("/")[-1]
    
    results_wide = col.query(query_texts=[qa["query"]], n_results=50)
    all_metas    = results_wide["metadatas"][0]
    
    target_rank = next(
        (i+1 for i, m in enumerate(all_metas)
         if m["file_path"].split("/")[-1] == target_file
         and m["function_name"] == target_sym),
        None
    )
    
    top10_docs  = results_wide["documents"][0][:10]
    referenced  = any(target_sym in doc for doc in top10_docs)
    
    print(f"{qa['source_symbol']}")
    print(f"  target rank : {target_rank or 'not found'}")
    print(f"  referenced in top-10 : {referenced}")

col, bm25, ids = col_orig, bm25_orig, ids_orig

PoissonSolver.solve
  target rank : 29
  referenced in top-10 : True
VariablePoissonSolver.solve
  target rank : not found
  referenced in top-10 : False
NonlinearVariablePoissonSolver.set_nonlinear_tolerance
  target rank : 4
  referenced in top-10 : True
DiffusionSolver.bdf3_step
  target rank : 3
  referenced in top-10 : True
PUDiffusionSolver.init
  target rank : not found
  referenced in top-10 : False
MultiSpeciesDiffusionSolver.bdf1_step
  target rank : 4
  referenced in top-10 : True
HeterogeneousMultiSpeciesDiffusionSolver.bdf1_step
  target rank : 49
  referenced in top-10 : False
PUSLAdvectionSolver.set_periodic_boundary
  target rank : 9
  referenced in top-10 : True
PUSLFDAdvectionDiffusionSolver.bdf1_step
  target rank : 1
  referenced in top-10 : True
PUSLPUAdvectionDiffusionSolver.init
  target rank : not found
  referenced in top-10 : False


In [17]:
# what does the added noise look like?
new_chunks = [c for c in all_chunks_v2 if (c["end_line"] - c["start_line"] + 1) < 5]
for c in new_chunks[:20]:
    lines = c["end_line"] - c["start_line"]
    print(f"{lines} lines | {c['class_name']}.{c['function_name']}")
    print("  " + c["text"][:120].replace("\n", "\n  "))
    print()

1 lines | DomainDescriptor.set_outer_level_set
  def set_outer_level_set(self, level_set: object) -> None:
          self.outer_level_set = level_set

1 lines | DomainDescriptor.set_sep_rad
  def set_sep_rad(self, sep_rad: float) -> None:
          self.sep_rad = float(sep_rad)

1 lines | DomainDescriptor.set_boundary_level_sets
  def set_boundary_level_sets(self, level_sets: list[object]) -> None:
          self.boundary_level_sets = level_sets

1 lines | DomainDescriptor.get_tree_points
  def get_tree_points(self, tree_mode: str) -> np.ndarray:
          return self._get_tree_data(tree_mode)[1]

1 lines | DomainDescriptor.get_interior_nodes
  def get_interior_nodes(self) -> np.ndarray:
          return self.xi

1 lines | DomainDescriptor.get_bdry_nodes
  def get_bdry_nodes(self) -> np.ndarray:
          return self.xb

1 lines | DomainDescriptor.get_ghost_nodes
  def get_ghost_nodes(self) -> np.ndarray:
          return self.xg

1 lines | DomainDescriptor.get_int_bdry_nodes
  def get

## v2 — Result

- **recall@3:** 1/10 — worse than v1 (2/10)
- **Root cause:** Lowering MIN_LINES adds noise to both retrieval legs
  simultaneously — insufficient text degrades embeddings, high keyword
  overlap with common terms (bdf, step, diffusion) pollutes BM25.
- **Recovery:** set_nonlinear_tolerance moved from not found to rank 4.
  Intended fix worked but was outweighed by regressions elsewhere.
- **Conclusion:** MIN_LINES is the wrong lever. Reverting to v1 for v3.

## v2 — Failure Mode Analysis

Ran wide retrieval (top-50, dense leg only) to diagnose regressions.

| Target | v1 rank | v2 rank | Change |
|--------|---------|---------|--------|
| PoissonSolver.solve | 15 | 29 | worse |
| VariablePoissonSolver.solve | 22 | not found | worse |
| NonlinearVariablePoissonSolver.set_nonlinear_tolerance | not found | 4 | recovered |
| DiffusionSolver.bdf3_step | 1 | 3 | slight drop |
| PUDiffusionSolver.init | not found | not found | same |
| MultiSpeciesDiffusionSolver.bdf1_step | 4 | 4 | same |
| HeterogeneousMultiSpeciesDiffusionSolver.bdf1_step | not found | 49 | appeared but buried |
| PUSLAdvectionSolver.set_periodic_boundary | 6 | 9 | worse |
| PUSLFDAdvectionDiffusionSolver.bdf1_step | 1 | 1 | same |
| PUSLPUAdvectionDiffusionSolver.init | 26 | not found | worse |

- bdf3_step sits at dense rank 3 — embeddings found it correctly — but
BM25 noise pushed it out of the top-3 RRF result. Both legs degrade
simultaneously. 
- One recovery, multiple regressions. Not worth it.

## v3 — LLM Summary Generation

Summaries were generated by prompting Codex with the full repo context.
Prompt used:

"Here is the repo for a scientific computing library called KernelPack-python.

For each function and method in src/kernelpack/solvers/, write a 1-2 sentence
plain English summary of what it does. Use the surrounding class and repo context
to inform your description. Do not invent behavior not present in the code.

Return as JSON: [{"function_name": "...", "class_name": "...", "summary": "..."}]

Save the result to ../kernelpack-rag-experiments/metadata/solvers_summaries.json"

Output: ../metadata/solvers_summaries.json

In [18]:
# build a lookup from the summaries JSON
with open("../metadata/solvers_summaries.json") as f:
    summaries = json.load(f)

# key = (function_name, class_name)
summary_lookup = {
    (s["function_name"], s["class_name"] or ""): s["summary"]
    for s in summaries
}

In [19]:
# v3: solvers chunks only, with summaries
all_chunks_v3 = [c for c in all_chunks if "solvers" in c["path"]]
print(f"v3 chunks: {len(all_chunks_v3)}")

v3 chunks: 149


In [20]:
# documents for kp-hybrid-v3 collection to only include solvers chunks, enriched with summaries
documents_v3 = []
for c in all_chunks_v3:
    key = (c["function_name"], c["class_name"] or "")
    summary = summary_lookup.get(key, "")
    enriched = f"{summary}\n\n{c['text']}" if summary else c["text"]
    documents_v3.append(enriched)

matched_v3 = sum(1 for c in all_chunks_v3
                 if (c["function_name"], c["class_name"] or "") in summary_lookup)
print(f"{matched_v3}/{len(all_chunks_v3)} chunks matched a summary")

149/149 chunks matched a summary


In [21]:
# v3a — solvers-only index, no summaries
# control experiment: isolates whether scoping to solvers alone improves recall
# independent of summary enrichment

col_v3a = client.get_or_create_collection("kp-hybrid-v3a", embedding_function=ef)
col_v3a.add(
    ids=[f"{c['path']}::{c['function_name']}::{c['start_line']}" for c in all_chunks_v3],
    documents=[c["text"] for c in all_chunks_v3],  # raw source, no summaries
    metadatas=[{
        "file_path":     str(Path(c["path"]).relative_to(REPO_PATH)),
        "function_name": c["function_name"],
        "class_name":    c["class_name"] or "",
        "start_line":    c["start_line"],
        "end_line":      c["end_line"],
    } for c in all_chunks_v3]
)

# BM25 on raw source only
bm25_v3a = BM25Okapi([c["text"].split() for c in all_chunks_v3])
ids_v3a  = [f"{c['path']}::{c['function_name']}::{c['start_line']}" for c in all_chunks_v3]

# eval
col_orig, bm25_orig, ids_orig = col, bm25, ids
col, bm25, ids = col_v3a, bm25_v3a, ids_v3a

hits_v3a   = {"api": 0, "workflow": 0, "conceptual": 0}
totals_v3a = {"api": 0, "workflow": 0, "conceptual": 0}
for qa in qa_pairs_solvers:
    results     = retrieve(qa["query"], silent=True)
    target_file = qa["source_file"].split("/")[-1]
    target_sym  = qa["source_symbol"].split(".")[-1]
    hit = any(
        r["file_path"].split("/")[-1] == target_file and
        r["function_name"] == target_sym
        for r in results
    )
    totals_v3a[qa["tier"]] += 1
    if hit:
        hits_v3a[qa["tier"]] += 1

col, bm25, ids = col_orig, bm25_orig, ids_orig

print("── v1 vs v3a vs v3 recall@3 ──────────")
print(f"  v1  (full index, no summaries)     : {sum(hits_v1.values())}/10")
print(f"  v3a (solvers only, no summaries)   : {sum(hits_v3a.values())}/10")

── v1 vs v3a vs v3 recall@3 ──────────
  v1  (full index, no summaries)     : 2/10
  v3a (solvers only, no summaries)   : 2/10


In [22]:
col_v3 = client.get_or_create_collection("kp-hybrid-v3", embedding_function=ef)
col_v3.add(
    ids=[f"{c['path']}::{c['function_name']}::{c['start_line']}" for c in all_chunks_v3],
    documents=documents_v3,
    metadatas=[{
        "file_path":     str(Path(c["path"]).relative_to(REPO_PATH)),
        "function_name": c["function_name"],
        "class_name":    c["class_name"] or "",
        "start_line":    c["start_line"],
        "end_line":      c["end_line"],
    } for c in all_chunks_v3]
)

In [23]:
# verify target functions are present in the solvers-only v3 index
results = col_v3.get(include=["metadatas"])
all_functions = [m["function_name"] for m in results["metadatas"]]

targets = [
    "set_nonlinear_tolerance",
    "set_periodic_boundary", 
    "set_linear_tolerance",
    "bdf1_step",
]

for target in targets:
    print(target, "→", "IN INDEX" if target in all_functions else "MISSING")

set_nonlinear_tolerance → MISSING
set_periodic_boundary → IN INDEX
set_linear_tolerance → MISSING
bdf1_step → IN INDEX


In [24]:
# sanity check: confirm summary is prepended to raw source
sample = col_v3.get(limit=1, include=["documents"])
print(sample["documents"][0][:500])

Translates a solver accuracy/order request and domain dimension into the stencil size, polynomial degree, spline degree, tree mode, and point set used by the RBF-FD assembler.

def build_stencil_properties(domain: DomainDescriptor, xi: int, theta: int, point_set: str) -> StencilProperties:
    # The solver layer asks for target order `xi` and operator order `theta`;
    # here we translate that request into a concrete stencil size and
    # polynomial degree that the rbffd layer understands.
   


In [25]:
# rebuild BM25 on v3 chunks
bm25_v3 = BM25Okapi([
    (summary_lookup.get((c["function_name"], c["class_name"] or ""), "") 
     + " " + c["text"]).split() 
    for c in all_chunks_v3
])
ids_v3 = [f"{c['path']}::{c['function_name']}::{c['start_line']}" for c in all_chunks_v3]

# swap to v3
col_orig, bm25_orig, ids_orig = col, bm25, ids
col, bm25, ids = col_v3, bm25_v3, ids_v3

# run eval
hits_v3 = {"api": 0, "workflow": 0, "conceptual": 0}
totals_v3 = {"api": 0, "workflow": 0, "conceptual": 0}
for qa in qa_pairs_solvers:
    results = retrieve(qa["query"], silent=True)
    target_file = qa["source_file"].split("/")[-1]
    target_sym  = qa["source_symbol"].split(".")[-1]
    hit = any(
        r["file_path"].split("/")[-1] == target_file and
        r["function_name"] == target_sym
        for r in results
    )
    totals_v3[qa["tier"]] += 1
    if hit:
        hits_v3[qa["tier"]] += 1

# restore
col, bm25, ids = col_orig, bm25_orig, ids_orig

print("── v1 vs v3a vs v3 recall@3 ──────────")
print(f"  v1  (full index, no summaries)     : {sum(hits_v1.values())}/10")
print(f"  v3a (solvers only, no summaries)   : {sum(hits_v3a.values())}/10")
print(f"  v3  (solvers only, with summaries) : {sum(hits_v3.values())}/10")

── v1 vs v3a vs v3 recall@3 ──────────
  v1  (full index, no summaries)     : 2/10
  v3a (solvers only, no summaries)   : 2/10
  v3  (solvers only, with summaries) : 3/10


In [26]:
col_orig, bm25_orig, ids_orig = col, bm25, ids

for qa in qa_pairs_solvers:
    target_file = qa["source_file"].split("/")[-1]
    target_sym  = qa["source_symbol"].split(".")[-1]
    
    # v1 result
    col, bm25, ids = col_orig, bm25_orig, ids_orig
    r1 = retrieve(qa["query"], silent=True)
    hit_v1 = any(
        r["file_path"].split("/")[-1] == target_file and
        r["function_name"] == target_sym
        for r in r1
    )
    
    # v3 result
    col, bm25, ids = col_v3, bm25_v3, ids_v3
    r3 = retrieve(qa["query"], silent=True)
    hit_v3 = any(
        r["file_path"].split("/")[-1] == target_file and
        r["function_name"] == target_sym
        for r in r3
    )
    
    col, bm25, ids = col_orig, bm25_orig, ids_orig
    
    if hit_v1 != hit_v3:
        status = "✓ -> ✗ REGRESSED" if hit_v1 else "✗ -> ✓ RECOVERED"
        print(f"{status}: {qa['source_symbol']}")

✗ -> ✓ RECOVERED: VariablePoissonSolver.solve
✗ -> ✓ RECOVERED: MultiSpeciesDiffusionSolver.bdf1_step
✓ -> ✗ REGRESSED: PUSLFDAdvectionDiffusionSolver.bdf1_step


In [27]:
# v2 vs benchmark
col_orig, bm25_orig, ids_orig = col, bm25, ids
col, bm25, ids = col_v2, bm25_v2, ids_v2

hits_bench_v2 = {"api": 0, "workflow": 0, "conceptual": 0}
totals_bench_v2 = {"api": 0, "workflow": 0, "conceptual": 0}
for qa in qa_pairs_benchmark:
    results = retrieve(qa["query"], silent=True)
    target_file = qa["source_file"].split("/")[-1]
    target_sym  = qa["source_symbol"].split(".")[-1]
    hit = any(
        r["file_path"].split("/")[-1] == target_file and
        r["function_name"] == target_sym
        for r in results
    )
    totals_bench_v2[qa["tier"]] += 1
    if hit:
        hits_bench_v2[qa["tier"]] += 1

col, bm25, ids = col_orig, bm25_orig, ids_orig

n = len(qa_pairs_benchmark)
print("── Benchmark recall@3 ────────────────")
print(f"  v1: {sum(hits_bench_v1.values())}/{n}")
print(f"  v2: {sum(hits_bench_v2.values())}/{n}")
print("  v3: n/a — solvers-only index, benchmark comparison not valid")

── Benchmark recall@3 ────────────────
  v1: 1/3
  v2: 1/3
  v3: n/a — solvers-only index, benchmark comparison not valid


In [28]:
col_orig, bm25_orig, ids_orig = col, bm25, ids
col, bm25, ids = col_v3, bm25_v3, ids_v3

for qa in qa_pairs_solvers:
    results = retrieve(qa["query"], silent=True)
    target_file = qa["source_file"].split("/")[-1]
    target_sym  = qa["source_symbol"].split(".")[-1]
    hit_v3 = any(
        r["file_path"].split("/")[-1] == target_file and
        r["function_name"] == target_sym
        for r in results
    )
    print(f"{'✓' if hit_v3 else '✗'} {qa['source_symbol']}")

col, bm25, ids = col_orig, bm25_orig, ids_orig

✗ PoissonSolver.solve
✓ VariablePoissonSolver.solve
✗ NonlinearVariablePoissonSolver.set_nonlinear_tolerance
✓ DiffusionSolver.bdf3_step
✗ PUDiffusionSolver.init
✓ MultiSpeciesDiffusionSolver.bdf1_step
✗ HeterogeneousMultiSpeciesDiffusionSolver.bdf1_step
✗ PUSLAdvectionSolver.set_periodic_boundary
✗ PUSLFDAdvectionDiffusionSolver.bdf1_step
✗ PUSLPUAdvectionDiffusionSolver.init


In [29]:
col_orig, bm25_orig, ids_orig = col, bm25, ids
col, bm25, ids = col_v3, bm25_v3, ids_v3

for qa in qa_pairs_solvers:
    target_sym  = qa["source_symbol"].split(".")[-1]
    target_file = qa["source_file"].split("/")[-1]
    
    results_wide = col.query(query_texts=[qa["query"]], n_results=50)
    all_metas    = results_wide["metadatas"][0]
    
    target_rank = next(
        (i+1 for i, m in enumerate(all_metas)
         if m["file_path"].split("/")[-1] == target_file
         and m["function_name"] == target_sym),
        None
    )
    
    top10_docs  = results_wide["documents"][0][:10]
    referenced  = any(target_sym in doc for doc in top10_docs)
    
    print(f"{qa['source_symbol']}")
    print(f"  target rank : {target_rank or 'not found'}")
    print(f"  referenced in top-10 : {referenced}")

col, bm25, ids = col_orig, bm25_orig, ids_orig

PoissonSolver.solve
  target rank : 8
  referenced in top-10 : True
VariablePoissonSolver.solve
  target rank : 5
  referenced in top-10 : True
NonlinearVariablePoissonSolver.set_nonlinear_tolerance
  target rank : not found
  referenced in top-10 : False
DiffusionSolver.bdf3_step
  target rank : 6
  referenced in top-10 : True
PUDiffusionSolver.init
  target rank : 36
  referenced in top-10 : True
MultiSpeciesDiffusionSolver.bdf1_step
  target rank : 2
  referenced in top-10 : True
HeterogeneousMultiSpeciesDiffusionSolver.bdf1_step
  target rank : not found
  referenced in top-10 : True
PUSLAdvectionSolver.set_periodic_boundary
  target rank : 5
  referenced in top-10 : True
PUSLFDAdvectionDiffusionSolver.bdf1_step
  target rank : 12
  referenced in top-10 : True
PUSLPUAdvectionDiffusionSolver.init
  target rank : 17
  referenced in top-10 : True


## v3 — Failure Mode Analysis

Ran wide retrieval (top-50, dense leg only) on all 10 eval pairs.
Ranks are dense-only proxies — actual hybrid RRF ranks will differ slightly.

| Target | v1 rank | v2 rank | v3 rank | v3 referenced in top-10 |
|--------|---------|---------|---------|--------------------------|
| PoissonSolver.solve | 15 | 29 | 8 | yes |
| VariablePoissonSolver.solve | 22 | not found | 5 | yes |
| NonlinearVariablePoissonSolver.set_nonlinear_tolerance | not found | 4 | not found | no |
| DiffusionSolver.bdf3_step | 1 | 3 | 6 | yes |
| PUDiffusionSolver.init | not found | not found | 36 | yes |
| MultiSpeciesDiffusionSolver.bdf1_step | 4 | 4 | 2 | yes |
| HeterogeneousMultiSpeciesDiffusionSolver.bdf1_step | not found | 49 | not found | yes |
| PUSLAdvectionSolver.set_periodic_boundary | 6 | 9 | 5 | yes |
| PUSLFDAdvectionDiffusionSolver.bdf1_step | 1 | 1 | 12 | yes |
| PUSLPUAdvectionDiffusionSolver.init | 26 | not found | 17 | yes |

## v3 — Results & Findings

### Recall@3
| Version | Change | recall@3 |
|---------|--------|----------|
| v1  | baseline — full index, raw source | 2/10 (20%) |
| v2  | MIN_LINES=1 | 1/10 (10%) |
| v3a | solvers only, no summaries (control) | 2/10 (20%) |
| v3  | solvers only, with summaries | 3/10 (30%) |

Note: v3a (solvers-only, no summaries) identical to v1.
Scoping the index to solvers only contributed nothing. The improvement
from v1 to v3 is attributable entirely to LLM summary enrichment.

### The real finding
Recall went up by 1. But the diagnostic tells the more interesting story.

- After summaries, 9/10 targets appear in the top-10 retrieved neighborhood.
The vocabulary gap is mostly closed. The bottleneck shifted.

- 2 recovered — VariablePoissonSolver.solve, MultiSpeciesDiffusionSolver.bdf1_step.
Summaries gave the embeddings something to match against. Direct fix.

- 1 regressed — PUSLFDAdvectionDiffusionSolver.bdf1_step dropped from rank 1
to rank 12. Summaries on neighboring chunks outscored it in RRF. Enriching
neighbors can hurt targets that were already ranking well on raw source alone.

### What's still broken
- Not indexed — set_nonlinear_tolerance, set_linear_tolerance are 2-line
  functions. Summaries can't help chunks that don't exist.
- Ranking — 9/10 targets in top-10 but outside top-3. This is no longer
  a retrieval problem. It's a reranking problem.
- Deep misses — PUDiffusionSolver.init (rank 36),
  HeterogeneousMultiSpeciesDiffusionSolver (not found) need cross-module
  context a solvers-only index can't provide.

### Next
A cross-encoder reranker on the top-10 candidates is the highest-leverage
next step. 9/10 targets already in that window. Everything else is noise
until the ranking problem is solved.